# Stage 4 — The portfolio you watch

**Checkpoint:** `stage-4`

Two more apps on the same spine: **early warning** (who is quietly getting worse?) and
**line increases** (who deserves more credit?). This is also where you meet an *honest gate* —
a metric that is **reported, not gated**, and the reason why matters more than the number.

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- stage guard: fail loudly and usefully, not mysteriously ---------------
NEEDS = [
    "ews/models/metadata.json",
    "line_increase/models/metadata.json",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-4 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + (Path("stage.txt").read_text().strip())
        + ". Fix with either:\n"
        "  git checkout stage-4      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-4 artifacts present.")

## 1. The watchlist — tiers, and the honest gate

In [ ]:
import json
from ews.src import watchlist as ews_watchlist

ews_meta = json.loads(Path("ews/models/metadata.json").read_text())
print("metrics:", json.dumps(ews_meta["metrics"], indent=2))
print("gate   :", json.dumps(ews_meta["gate"], indent=2))

ews = ews_watchlist.score_population()
tiers = ews["risk_tier"].value_counts().reindex(["High", "Medium", "Low"]).fillna(0)
tiers.to_frame("accounts")

> **Why the AUC here is reported, not gated.** Deterioration is a noisier target than default,
> and the data-generating process injects that noise deliberately. Rather than demand an AUC the
> data cannot support, this module is gated on what actually matters operationally: **how much of
> tomorrow's trouble lands in the top decile you can actually work.**

In [ ]:
COLT = {"High": "#b3372e", "Medium": "#c9a227", "Low": "#2f7d4f"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(tiers.index, tiers.values, color=[COLT[t] for t in tiers.index])
for i, v in enumerate(tiers.values):
    axes[0].text(i, v + 60, f"{int(v):,}", ha="center", weight="bold")
axes[0].set_title("Watchlist tiers"); axes[0].set_ylabel("accounts")

axes[1].hist(ews["prob"], bins=50, color="#9fb4d4", edgecolor="white")
axes[1].set_xlabel("p(deterioration)"); axes[1].set_title("Deterioration probability")
plt.tight_layout(); plt.show()

## 2. Named triggers — reasons, not vibes

In [ ]:
from collections import Counter
trig = Counter(t for ts in ews["triggers"] for t in ts)
s = pd.Series(dict(trig)).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(s.index, s.values, color="#5b8ac6")
ax.set_xlabel("accounts flagged"); ax.set_title("How often each named trigger fires")
plt.tight_layout(); plt.show()

top10 = (ews.sort_values("prob", ascending=False).head(10)
         [["business_id", "risk_tier", "prob", "triggers"]])
top10["triggers"] = top10["triggers"].apply(lambda t: ", ".join(t))
top10.reset_index(drop=True).style.format({"prob": "{:.1%}"})

## 3. Does the watchlist concentrate the trouble?

This is the gated metric — and the only one that matters operationally: **if you can only work
10% of the book, how much of tomorrow's trouble is in that 10%?**

In [ ]:
cap  = ews_meta["metrics"]["top_decile_capture"]
lift = ews_meta["metrics"]["top_decile_lift"]
gate = ews_meta["gate"]["capture_min"]

print(f"top-decile capture (held out): {cap:.1%}")
print(f"                      = lift : {lift:.2f}x random")
print(f"gate                         : >= {gate:.0%}")
print(f"result                       : {'PASS' if cap >= gate else 'FAIL'}")
print(f"\nWork 10% of the book, catch {cap:.1%} of the deteriorations.")

### The curve — and a warning about reading it

Below we draw the capture curve over the **whole book**, because that is the population you
actually operate on. But the model was trained on most of these rows, so this curve is
**optimistic**. The honest number is the held-out one above.

That gap is the lesson: an in-sample curve will always flatter you. Quote the held-out
number in the memo.

In [ ]:
from shared.config import RAW
port = pd.read_parquet(RAW / "portfolio.parquet").set_index("business_id")
truth = port["deterioration_next_6_12mo"].reindex(ews["business_id"]).to_numpy()

order = np.argsort(-ews["prob"].to_numpy())
y = truth[order]
xs = np.linspace(.05, 1, 20)
curve = [y[:max(1, int(len(y) * q))].sum() / y.sum() for q in xs]
in_sample_10 = y[:int(len(y) * .1)].sum() / y.sum()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(xs * 100, np.array(curve) * 100, marker="o", color="#245bb2", label="in-sample (optimistic)")
ax.plot([0, 100], [0, 100], ls="--", color="#999", label="random")
ax.scatter([10], [cap * 100], s=90, color="#b3372e", zorder=5,
           label=f"held-out at 10% = {cap:.1%}")
ax.axvline(10, color="#b3372e", ls=":", lw=1.2)
ax.set_xlabel("% of book reviewed (worst first)"); ax.set_ylabel("% of deteriorations caught")
ax.set_title("Capture curve — quote the red dot, not the blue line")
ax.legend(); plt.tight_layout(); plt.show()

print(f"in-sample at 10%: {in_sample_10:.1%}   vs   held-out: {cap:.1%}")
print("The difference is why we gate on the held-out number.")

## 4. Line increases — growth inside risk appetite

In [ ]:
from line_increase.src import candidates as li_candidates

li_meta = json.loads(Path("line_increase/models/metadata.json").read_text())
li = li_candidates.score_population()
# An offer is what the platform calls an offer: `eligible`. This is exactly the
# filter line_increase.src.candidates.candidates() applies, so the notebook and
# the portal can never disagree.
offers = li[li["eligible"]]

print(f"on book, scored       : {len(li):,}")
print(f"offers made           : {len(offers):,}  ({len(offers)/len(li):.1%} of the book)")
print(f"total incremental line: ${offers['recommended_amount'].sum():,.0f}")
print(f"median incremental ROE: {offers['incremental_roe'].median():.1%}")
print(f"clears the hurdle     : {offers['clears_hurdle'].mean():.1%} of offers")
li.head()

In [ ]:
cols = ["recommended_amount", "incremental_roe", "pd"]
if len(offers) and cols:
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3.6))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cols):
        ax.hist(offers[c], bins=35, color="#5b8ac6", edgecolor="white")
        ax.set_title(c)
    plt.tight_layout(); plt.show()

    print("The cohort test — is the cohort we grow SAFER than the book?")
    print(f"  offered cohort mean PD  : {offers['pd'].mean():.4f}")
    print(f"  whole book mean PD      : {li['pd'].mean():.4f}")
    print("  " + ("PASS — growth is going to the safer end of the book."
                  if offers['pd'].mean() < li['pd'].mean() else "FAIL — investigate."))
    print()
    print("  ...and are they people who will USE the line?")
    print(f"  offered cohort utilization: {offers['utilization_onbook'].mean():.4f}")
    print(f"  whole book utilization    : {li['utilization_onbook'].mean():.4f}")
    print("\n  Safer AND more utilised — that combination is the whole point:")
    print("  growth inside risk appetite, not growth by loosening it.")
else:
    print("No offers in this run.")

---
**Next:** `05_stage5_governance.ipynb` — assemble everything into something a validator could
read, and defend it.